# model-4: ConvNeXtTiny

Pasangan: **Ilham** (trial 1-5) dan **Arta** (trial 6-10).  
Ide kunci: **Arsitektur CNN Modern (Kernel 7x7 Depthwise, Inverted Bottleneck, LayerNorm, GELU)**.  
Dibuat dari template notebook Kaggle kelompok Neuro Computing (M11), PENS.  
Versi 2 (8 November 2026): data dibaca dari dataset Kaggle `simplicity-224-letterbox`.

**Pengaturan Kaggle (wajib):**
- *Settings > Accelerator*: GPU (T4 atau P100)
- *Settings > Internet*: **ON**, untuk `git clone` dan unduh bobot awal ImageNet (butuh verifikasi nomor HP di Kaggle)
- *Add Input* (panel kanan): cari dataset `simplicity-224-letterbox` milik `yonobengkel12`, lalu klik tombol tambah (+). Isinya gambar 224x224 yang sudah di-letterbox dan identik dengan hasil `data_utils.py`.
- Untuk pelatihan stabil, gunakan *Save Version > Save & Run All* agar proses tetap berjalan walau browser ditutup.

**Aturan Main:**
1. Sel bertanda `JANGAN DIUBAH` dipakai apa adanya. `common/data_utils.py` dikunci dan hanya dipakai lewat import.
2. Di sel *Pengaturan pasangan* (bagian 4), tentukan `NAMA = "arta"` dan `TRIAL_SAYA = [6, 7, 8, 9, 10]`.
3. Dari tab *Output Kaggle*, unduh `trials_<nama>.csv` dan `history_<nama>.json`, lalu masukkan ke folder `model-4_convnext_tiny/`. Berkas `.keras` model terbaik disimpan di output Kaggle, jangan di-commit ke Git.
4. Setelah kedua orang selesai, sel *Gabung hasil* menulis `trials.csv` dan `history_convnext_tiny.png`.
5. Evaluasi test set (`JALANKAN_TEST = True`) hanya dijalankan satu kali di akhir oleh pemilik trial terbaik.


## 1. Setup bersama (JANGAN DIUBAH)
Clone repo (di Kaggle) atau cari root repo (di lokal), import `data_utils`, cetak sidik versinya, lalu cek perangkat.  
Sidik yang tercetak harus sama dengan sidik resmi di README bagian 3 (per 8 November 2026: `b3e0629783c5`).


In [ ]:
# ===== SETUP BERSAMA: JANGAN DIUBAH =====
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/0Raja0-RA/simplicity-cnn-comparison.git"
BRANCH = "main"

if Path("/kaggle/working").exists():                      # di Kaggle
    ROOT = Path("/kaggle/working/simplicity-cnn-comparison")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", "--branch", BRANCH, REPO_URL, str(ROOT)], check=True)
else:                                                     # di lokal: naik sampai ketemu root repo
    ROOT = Path.cwd().resolve()
    while not (ROOT / "common" / "data_utils.py").exists():
        if ROOT.parent == ROOT:
            raise FileNotFoundError("Root repo (berisi common/data_utils.py) tidak ditemukan.")
        ROOT = ROOT.parent

sys.path.insert(0, str(ROOT))
from common import data_utils as du

SIDIK = du.sidik_config()
INFO_PERANGKAT = du.cek_perangkat()
print("ROOT:", ROOT)


## 2. Data (JANGAN DIUBAH, kecuali `BATCH_SIZE`)
Gambar float32 0-255 berukuran `du.IMG_SIZE`, sudah di-padding, belum dinormalisasi. Label integer 0-9
(pakai `sparse_categorical_crossentropy`). Augmentasi **tidak** ada di dataset; pasang `du.get_augmentation()` di model.
`du.load_data()` mencetak sumber data. Di Kaggle harus tertulis `simplicity_224.npz (npz dataset Kaggle)`; sel ini berhenti dengan pesan galat kalau dataset belum dipasang.


In [ ]:
BATCH_SIZE = 32          # boleh diubah per pasangan (README bagian 5)

# Di Kaggle, dataset simplicity-224-letterbox wajib terpasang (Add Input) supaya semua pasangan memakai gambar yang sama persis
if Path("/kaggle/working").exists() and not any(Path("/kaggle/input").rglob(du.NPZ_NAMA)):
    raise RuntimeError(f"Dataset Kaggle {du.KAGGLE_DATASET} belum dipasang. "
                       "Tambahkan lewat Add Input di panel kanan, lalu jalankan ulang sel ini.")

train_ds, val_ds = du.load_data(batch_size=BATCH_SIZE)   # mencetak sumber data: harus simplicity_224.npz
CLASS_NAMES = du.get_class_names()
print("IMG_SIZE:", du.IMG_SIZE, "| kelas:", CLASS_NAMES)


## 3. Model (diisi pasangan): ConvNeXtTiny
**Ide kunci: Modern Pure CNN (Liu et al., 2022).**  
ConvNeXt mengadopsi prinsip desain arsitektur modern Vision Transformer (ViT) tanpa mekanisme self-attention:
1. **Macro design**: Rasio stage 1:1:3:1 (proporsi mirip Swin Transformer) dan inverted bottleneck (dimensi kanal mengembang 4x di dalam block).
2. **Depthwise convolution 7x7**: Menggantikan standar 3x3 untuk memperluas receptive field lokal tanpa lonjakan parameter.
3. **Layer Normalization (LN)**: Menggantikan Batch Normalization (BN) sehingga model sangat stabil pada batch kecil (16 atau 32) dan tidak bergantung pada statistik mini-batch.
4. **Fewer activations & normalizations**: Menggunakan aktivasi **GELU** (bukan ReLU) dan hanya dipasang satu kali per block (mengikuti desain Transformer Block).

**Normalisasi Citra:**
ConvNeXt di Keras (`keras.applications.ConvNeXtTiny`) sudah memiliki layer `convnext_tiny_prestem_normalization` di dalam dirinya. Layer ini otomatis memetakan piksel input `0-255` ke normalisasi ImageNet mean/std. Karena itu, sesuai aturan nomor 5 pada panduan kelompok, kita tidak perlu menambahkan layer normalisasi tambahan di luar model.

**Urutan dalam model:**  
`Input(224, 224, 3)` -> `du.get_augmentation()` -> `ConvNeXtTiny` (ImageNet, `include_top=False`, `pooling="avg"`) -> `Dropout` -> (opsional `Dense GELU` + `Dropout`) -> `Dense 10 softmax`.


In [ ]:
from tensorflow import keras

# Strategi pembukaan layer ConvNeXtTiny per stage
# ConvNeXtTiny memiliki 4 stage utama: stage 0, stage 1, stage 2, stage 3.
# Layer akhir sebelum pooling adalah layer_normalization.
AWALAN_DIBUKA = {
    "beku": (),                                                                          # hanya head klasifikasi yang dilatih
    "stage_3": ("convnext_tiny_stage_3_", "layer_normalization"),                        # buka stage 3 (blok konvolusi teratas) + final LN
    "stage_2+3": ("convnext_tiny_stage_2_", "convnext_tiny_stage_3_", "layer_normalization"), # buka stage 2 dan 3
}


def bangun_model(cfg):
    """ConvNeXtTiny + classification head 10 kelas sesuai konfigurasi trial, sudah di-compile."""
    du.set_seed()                                   # bobot head awal sama di akun mana pun
    inputs = keras.Input(shape=du.IMG_SIZE + (3,))
    x = du.get_augmentation()(inputs)               # wajib, aktif hanya saat training

    # ConvNeXtTiny di Keras sudah menormalisasi piksel [0, 255] secara internal
    # melalui layer 'convnext_tiny_prestem_normalization'. Sesuai aturan 5 panduan kelompok,
    # tidak ada normalisasi tambahan di luar model.
    base = keras.applications.ConvNeXtTiny(
        include_top=False,
        weights="imagenet",
        input_shape=du.IMG_SIZE + (3,),
        pooling="avg"
    )

    awalan = AWALAN_DIBUKA[cfg["buka_layer"]]
    base.trainable = bool(awalan)
    for layer in base.layers:
        layer.trainable = (bool(awalan) and layer.name.startswith(awalan))
    x = base(x, training=False)

    x = keras.layers.Dropout(cfg["dropout"])(x)
    if cfg["unit_head"] > 0:
        x = keras.layers.Dense(cfg["unit_head"], activation="gelu")(x)
        x = keras.layers.Dropout(cfg["dropout"])(x)
    outputs = keras.layers.Dense(len(CLASS_NAMES), activation="softmax")(x)
    model = keras.Model(inputs, outputs, name="convnext_tiny_simplicity")

    if cfg["optimizer"] == "adamw":
        opt = keras.optimizers.AdamW(learning_rate=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    else:
        opt = keras.optimizers.Adam(learning_rate=cfg["learning_rate"])

    model.compile(optimizer=opt, loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return model


## 4. Tuning: random search (diisi pasangan)
Ikuti README bagian 5: **10 trial**, maksimal **40 epoch**, `EarlyStopping(val_loss, patience=8, restore_best_weights=True)`.  
Pilih konfigurasi terbaik dari **validation** saja (val_acc tertinggi; kalau seri, val_loss terendah).

**Pembagian trial:** ke-10 konfigurasi diundi sekaligus di awal dengan `seed=42`, sehingga daftarnya sama persis di akun Ilham dan Arta.
- **Ilham**: menjalankan trial 1-5
- **Arta**: menjalankan trial 6-10  
Hasil tiap orang dicatat di `trials_<nama>.csv`. Trial yang sudah selesai akan dilewati otomatis, sehingga jika sesi Kaggle terputus, proses dapat melanjutkan tanpa mengulang dari awal.


In [ ]:
# ===== PENGATURAN PASANGAN: hanya dua baris ini yang disesuaikan =====
NAMA = "arta"                    # "ilham" (trial 1-5) atau "arta" (trial 6-10)
TRIAL_SAYA = [6, 7, 8, 9, 10]    # Ilham: [1, 2, 3, 4, 5]   Arta: [6, 7, 8, 9, 10]
# ======================================================================

import csv
import hashlib
import json
import shutil
import time

import numpy as np
import pandas as pd

ARSITEKTUR = "convnext_tiny"
N_TRIAL = 10                     # README bagian 5
MAX_EPOCH = 40
PATIENCE = 8
assert set(TRIAL_SAYA) <= set(range(1, N_TRIAL + 1)), "TRIAL_SAYA harus di antara 1-10"

# Folder pasangan di repo: model-4_convnext_tiny (nama lama model-4 tetap didukung)
if (ROOT / "model-4_convnext_tiny").is_dir():
    MODEL_DIR = ROOT / "model-4_convnext_tiny"
elif (ROOT / "model-4").is_dir():
    MODEL_DIR = ROOT / "model-4"
else:
    MODEL_DIR = ROOT / "model-4_convnext_tiny"
    MODEL_DIR.mkdir(exist_ok=True)

TRIALS_SAYA_CSV = MODEL_DIR / f"trials_{NAMA}.csv"
HISTORY_SAYA_JSON = MODEL_DIR / f"history_{NAMA}.json"

# Berkas .keras disimpan DI LUAR folder repo (aturan 9): /kaggle/working di Kaggle
SIMPAN_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else ROOT.parent / "model_tersimpan_simplicity"
SIMPAN_DIR.mkdir(exist_ok=True)

# Melanjutkan sesi: jika trials_<nama>.csv / history_<nama>.json diunggah lewat Add Input, salin ke folder model
if Path("/kaggle/input").exists():
    for berkas in (TRIALS_SAYA_CSV, HISTORY_SAYA_JSON):
        if not berkas.exists():
            temuan = sorted(Path("/kaggle/input").rglob(berkas.name))
            if temuan:
                shutil.copy(temuan[0], berkas)
                print("Dilanjutkan dari:", temuan[0])

print(f"{NAMA} menjalankan trial {TRIAL_SAYA}")
print("Log trial :", TRIALS_SAYA_CSV)
print("Model     :", SIMPAN_DIR)


In [ ]:
# Ruang pencarian ConvNeXtTiny. Diundi sekali untuk ke-10 trial dengan seed 42.
#  - buka_layer : membuka stage konvolusi teratas (stage 3 atau stage 2+3)
#  - learning_rate (log-uniform): mode beku boleh lebih besar (1e-4 .. 3e-3); mode unfreeze kecil (1e-5 .. 2e-4)
#  - weight_decay: 1e-5, 1e-4, 1e-3 untuk AdamW (0.0 untuk Adam)
#  - dropout 0.2-0.5: mencegah overfitting pada dataset kecil (500 gambar train)
#  - unit_head: 0 (pooling langsung ke softmax) atau 256 unit intermediate GELU
#  - batch_size: 16 atau 32
def sampel_konfigurasi(rng):
    buka = str(rng.choice(["beku", "stage_3", "stage_2+3"]))
    if buka == "beku":
        lr = 10 ** rng.uniform(np.log10(1e-4), np.log10(3e-3))
    else:
        lr = 10 ** rng.uniform(np.log10(1e-5), np.log10(2e-4))
    optimizer = str(rng.choice(["adam", "adamw"]))
    weight_decay = float(rng.choice([1e-5, 1e-4, 1e-3])) if optimizer == "adamw" else 0.0
    return {
        "buka_layer": buka,
        "learning_rate": float(f"{lr:.2e}"),
        "optimizer": optimizer,
        "weight_decay": weight_decay,
        "dropout": round(float(rng.uniform(0.2, 0.5)), 2),
        "unit_head": int(rng.choice([0, 256])),
        "batch_size": int(rng.choice([16, 32])),
    }


_rng = np.random.default_rng(du.SEED)
KONFIGURASI = {t: sampel_konfigurasi(_rng) for t in range(1, N_TRIAL + 1)}
PARAMETER = list(KONFIGURASI[1])

SIDIK_KONFIG = hashlib.sha256(json.dumps(KONFIGURASI, sort_keys=True).encode()).hexdigest()[:12]
print("Sidik konfigurasi:", SIDIK_KONFIG, "(harus sama di akun Ilham dan Arta)")
pd.DataFrame.from_dict(KONFIGURASI, orient="index").rename_axis("trial")


In [ ]:
KOLOM = ["trial", *PARAMETER, "val_acc", "val_loss", "best_epoch", "epoch_dijalankan",
         "train_time_sec", "dijalankan_oleh", "sidik_config"]


def baca_trials(path):
    if not path.exists():
        return []
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))


def skor(row):
    """Urutan pemilihan: val_acc tertinggi, kalau seri val_loss terendah."""
    return (float(row["val_acc"]), -float(row["val_loss"]))


riwayat_saya = json.loads(HISTORY_SAYA_JSON.read_text(encoding="utf-8")) if HISTORY_SAYA_JSON.exists() else {}
log_saya = baca_trials(TRIALS_SAYA_CSV)
selesai = {int(r["trial"]) for r in log_saya}
terbaik_saya = max(log_saya, key=skor) if log_saya else None

for t in TRIAL_SAYA:
    if t in selesai:
        print(f"Trial {t}: sudah tercatat, dilewati")
        continue
    cfg = KONFIGURASI[t]
    print(f"\n===== Trial {t}/{N_TRIAL}: {cfg}")

    keras.backend.clear_session()
    train_ds, val_ds = du.load_data(batch_size=cfg["batch_size"])   # juga memanggil set_seed(42)
    model = bangun_model(cfg)
    es = keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE,
                                       restore_best_weights=True, verbose=1)
    mulai = time.time()
    hist = model.fit(train_ds, validation_data=val_ds, epochs=MAX_EPOCH, callbacks=[es], verbose=2)
    durasi = time.time() - mulai

    val_loss, val_acc = model.evaluate(val_ds, verbose=0)           # bobot terbaik (sudah di-restore)
    row = {"trial": t, **cfg,
           "val_acc": round(float(val_acc), 4), "val_loss": round(float(val_loss), 4),
           "best_epoch": int(np.argmin(hist.history["val_loss"])) + 1,
           "epoch_dijalankan": len(hist.history["val_loss"]),
           "train_time_sec": round(durasi, 1), "dijalankan_oleh": NAMA, "sidik_config": SIDIK[:12]}

    baru = not TRIALS_SAYA_CSV.exists()
    with open(TRIALS_SAYA_CSV, "a", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=KOLOM, lineterminator="\n")
        if baru:
            w.writeheader()
        w.writerow(row)
    riwayat_saya[str(t)] = {k: [round(float(v), 5) for v in vals] for k, vals in hist.history.items()}
    HISTORY_SAYA_JSON.write_text(json.dumps(riwayat_saya, indent=1), encoding="utf-8")
    print(f"Trial {t}: val_acc {row['val_acc']:.4f}, val_loss {row['val_loss']:.4f}, "
          f"epoch terbaik {row['best_epoch']}, {durasi / 60:.1f} menit")

    # Simpan hanya model terbaik milik sendiri (di luar repo), untuk evaluasi test nanti
    row_str = {k: str(v) for k, v in row.items()}
    if terbaik_saya is None or skor(row_str) > skor(terbaik_saya):
        if terbaik_saya is not None:
            (SIMPAN_DIR / f"model-4_trial{int(terbaik_saya['trial']):02d}.keras").unlink(missing_ok=True)
        model.save(SIMPAN_DIR / f"model-4_trial{t:02d}.keras")
        terbaik_saya = row_str
        print(f"Model trial {t} disimpan sebagai model terbaik milik {NAMA}")

pd.read_csv(TRIALS_SAYA_CSV)


### Gabung hasil pasangan
Membaca semua `trials_*.csv` di folder `model-4_convnext_tiny/`. Kalau kesepuluh trial sudah lengkap, sel ini menulis `trials.csv`
dan `history_convnext_tiny.png` (grafik konvergensi gradient descent: loss dan akurasi train vs validation dari trial terbaik).  
Sel ini tidak butuh GPU, jadi boleh dijalankan di lokal setelah `git pull` atau menggabungkan berkas.


In [ ]:
import matplotlib.pyplot as plt

semua = {}
for p in sorted(MODEL_DIR.glob("trials_*.csv")):
    for r in baca_trials(p):
        semua[int(r["trial"])] = r
riwayat = {}
for p in sorted(MODEL_DIR.glob("history_*.json")):
    if p.name != f"history_{ARSITEKTUR}.json":
        riwayat.update(json.loads(p.read_text(encoding="utf-8")))

kurang = sorted(set(range(1, N_TRIAL + 1)) - set(semua))
lengkap = not kurang
print("Trial tercatat:", sorted(semua), "| belum:", kurang if kurang else "-")

if semua:
    df = pd.DataFrame([semua[t] for t in sorted(semua)])
    terbaik = max(semua.values(), key=skor)
    t_best = int(terbaik["trial"])
    print(f"\nTrial terbaik{'' if lengkap else ' (SEMENTARA)'}: {t_best} oleh {terbaik['dijalankan_oleh']}, "
          f"val_acc {float(terbaik['val_acc']):.4f}, val_loss {float(terbaik['val_loss']):.4f}")
    print("Konfigurasi:", {k: terbaik[k] for k in PARAMETER})

    h = riwayat.get(str(t_best))
    if h:
        ep = range(1, len(h["loss"]) + 1)
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        for i, (m, judul) in enumerate([("loss", "Loss"), ("accuracy", "Akurasi")]):
            ax[i].plot(ep, h[m], label="train", color="#1f77b4", lw=2)
            ax[i].plot(ep, h[f"val_{m}"], label="validation", color="#ff7f0e", lw=2)
            ax[i].axvline(int(terbaik["best_epoch"]), color="gray", ls="--", lw=1.5, label="epoch terbaik")
            ax[i].set_title(f"{judul} ConvNeXtTiny (Trial {t_best})", fontsize=12, fontweight="bold")
            ax[i].set_xlabel("Epoch", fontsize=10)
            ax[i].set_ylabel(judul, fontsize=10)
            ax[i].grid(True, linestyle=":", alpha=0.6)
            ax[i].legend(fontsize=9)
        fig.tight_layout()
        if lengkap:
            fig.savefig(MODEL_DIR / f"history_{ARSITEKTUR}.png", dpi=150)
            print("Grafik disimpan ke:", MODEL_DIR / f"history_{ARSITEKTUR}.png")
        plt.show()

    if lengkap:
        df.to_csv(MODEL_DIR / "trials.csv", index=False, lineterminator="\n")
        print("Ditulis:", MODEL_DIR / "trials.csv", "dan", MODEL_DIR / f"history_{ARSITEKTUR}.png")
        print("Langkah berikutnya: tulis konfigurasi terbaik di model-4_convnext_tiny/README.md, commit, baru evaluasi test.")
    display(df)


## 5. Evaluasi test (SEL PALING AKHIR)
Jalankan **satu kali** saja, setelah konfigurasi terbaik ditulis dan di-commit di `model-4_convnext_tiny/README.md`.  
Ubah `JALANKAN_TEST = True` hanya pada saat itu. Hasil test tidak boleh dipakai untuk mengubah hyperparameter apa pun.  
Dijalankan oleh **pemilik trial terbaik** (lihat sel *Gabung hasil*), karena model terbaiknya ada di Output Kaggle
miliknya (`model-4_trialXX.keras`). Kalau sesinya sudah berganti, unggah berkas itu lewat *Add Input* terlebih dahulu.


In [ ]:
JALANKAN_TEST = False

if JALANKAN_TEST:
    assert (MODEL_DIR / "trials.csv").exists(), "trials.csv belum ada: tuning belum lengkap"
    terbaik = max(baca_trials(MODEL_DIR / "trials.csv"), key=skor)
    t_best = int(terbaik["trial"])
    nama_model = f"model-4_trial{t_best:02d}.keras"
    kandidat = [SIMPAN_DIR / nama_model]
    if Path("/kaggle/input").exists():
        kandidat += sorted(Path("/kaggle/input").rglob(nama_model))
    ada = [p for p in kandidat if p.exists()]
    if not ada:
        raise FileNotFoundError(f"{nama_model} tidak ditemukan. Jalankan sel ini di akun {terbaik['dijalankan_oleh']}, "
                                "atau unggah berkas itu dari Output Kaggle lewat Add Input.")
    path_model = ada[0]
    model_terbaik = keras.models.load_model(path_model)
    print("Model:", path_model, "| trial", t_best, {k: terbaik[k] for k in PARAMETER})

    test_ds = du.load_test()
    y_true = du.ambil_label(test_ds)
    model_terbaik.predict(test_ds.take(1), verbose=0)                 # pemanasan GPU, tidak diukur
    mulai = time.time()
    y_pred = model_terbaik.predict(test_ds, verbose=0).argmax(1)
    inferensi_ms = (time.time() - mulai) / len(y_true) * 1000

    cm = np.zeros((len(CLASS_NAMES),) * 2, dtype=int)
    np.add.at(cm, (y_true, y_pred), 1)                                 # baris = asli, kolom = prediksi
    tp = np.diag(cm)
    presisi = np.divide(tp, cm.sum(0), out=np.zeros(len(tp)), where=cm.sum(0) > 0)
    recall = np.divide(tp, cm.sum(1), out=np.zeros(len(tp)), where=cm.sum(1) > 0)
    f1 = np.divide(2 * presisi * recall, presisi + recall, out=np.zeros(len(tp)), where=(presisi + recall) > 0)

    hasil = {
        "arsitektur": "ConvNeXtTiny", "trial_terbaik": t_best,
        "test_acc": round(float((y_true == y_pred).mean()), 4),
        "macro_f1": round(float(f1.mean()), 4),
        "macro_precision": round(float(presisi.mean()), 4),
        "macro_recall": round(float(recall.mean()), 4),
        "jumlah_parameter": int(model_terbaik.count_params()),
        "parameter_dilatih": int(sum(np.prod(w.shape) for w in model_terbaik.trainable_weights)),
        "ukuran_model_mb": round(path_model.stat().st_size / 2**20, 1),
        "inferensi_ms_per_gambar": round(inferensi_ms, 2),
        "train_time_sec": float(terbaik["train_time_sec"]),
        "sidik_config": SIDIK[:12],
    }
    (MODEL_DIR / "hasil_test.json").write_text(json.dumps(hasil, indent=2), encoding="utf-8")
    print(json.dumps(hasil, indent=2))
    display(pd.DataFrame({"presisi": presisi, "recall": recall, "f1": f1}, index=CLASS_NAMES).round(3))

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(CLASS_NAMES)), CLASS_NAMES, rotation=45, ha="right")
    ax.set_yticks(range(len(CLASS_NAMES)), CLASS_NAMES)
    for i in range(len(CLASS_NAMES)):
        for j in range(len(CLASS_NAMES)):
            ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set(xlabel="prediksi", ylabel="asli", title=f"Confusion matrix ConvNeXtTiny (test, akurasi {hasil['test_acc']:.1%})")
    fig.tight_layout()
    fig.savefig(MODEL_DIR / f"confusion_{ARSITEKTUR}.png", dpi=120)
    plt.show()
else:
    print("Evaluasi test belum dijalankan (JALANKAN_TEST = False).")
